In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load insurance data
df = pd.read_csv('../data/raw/insurance_claims.csv')
print(f'Shape of dataset: {df.shape}')
print(df.dtypes)

In [ ]:
# Fraud reported dist
fraud_pct = df['fraud_reported'].value_counts(normalize=True)*100
print(fraud_pct)
fraud_pct.plot(kind='bar')
plt.title('Fraud Reported Distribution')
plt.ylabel('Percentage')
plt.show()

In [ ]:
# By incident type
plt.figure(figsize=(10, 5))
sns.countplot(x='incident_type', hue='fraud_reported', data=df)
plt.title('Fraud by Incident Type')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Age dist
plt.figure(figsize=(8, 5))
sns.histplot(df[df['fraud_reported']=='N']['age'], color='blue', label='Normal', kde=True, alpha=0.5)
sns.histplot(df[df['fraud_reported']=='Y']['age'], color='red', label='Fraud', kde=True, alpha=0.5)
plt.title('Age Distribution: Fraud vs Normal')
plt.legend()
plt.show()

In [ ]:
# Correlation for numerical
corr = df.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='viridis')
plt.title('Numeric Features Heatmap')
plt.show()

In [ ]:
# Missing values
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, cmap='magma')
plt.title('Missing Value Map')
plt.show()

In [ ]:
# Chi2 prep for Top features
from sklearn.feature_selection import chi2
from sklearn.preprocessing import LabelEncoder

# simple transform
cat_cols = df.select_dtypes(include=['object']).columns
le = LabelEncoder()
df_encoded = df.copy()
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

X = df_encoded.drop(columns=['fraud_reported'])
y = df_encoded['fraud_reported']

# Ensure only non-missing / non-negative
X = X.fillna(0)
for col in X.columns:
    if X[col].min() < 0:
        X = X.drop(columns=[col])

chi_scores = chi2(X, y)
p_values = pd.Series(chi_scores[1], index=X.columns)
p_values.sort_values(ascending=True)[:10].plot(kind='bar', color='green')
plt.title('Top Categorical Features (Chi2 Test - Lowest P-values)')
plt.show()